# Õppetund 13 - Agendi Mälu


## Seadistamine

See märkmik demonstreerib, kuidas luua reisibroneerimise agent **püsiva mäluga** kasutades **Microsoft Agent Frameworki** (MAF).

Õpid, kuidas erinevat tüüpi agendi mälu — töömälu, lühiajaline ja pikaajaline — mõjutavad seda, kuidas agent hoiab ja kasutab teavet vestluste jooksul.

**Eeltingimused:**
- Azure AI Foundry projekt koos kasutusele võetud vestlusmudeliga (näiteks `gpt-4o-mini`).
- Sisselogitud Azure CLI abil — käivita oma terminalis `az login`.
- `AZURE_AI_PROJECT_ENDPOINT` — sinu Azure AI Foundry projekti lõpp-punkt.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — kasutusele võetud mudeli nimi.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Agendi mälutüübid

Tehisintellekti agendid saavad kasutada erinevat tüüpi mälu, millest igaüks täidab oma kindlat eesmärki:

### Töömälus
Vestlussõnumite jada — sõnumid, mis vahetatakse ühe seansi jooksul. Agent võib viidata varasematele sõnumitele samas vestlussõnumite jadast, et säilitada põhjendatus. MAF-is luuakse see kaudu **`agent.create_session()`**, mis tagastab `AgentSession`.

### Lühiajaline mälu
Teave, mis püsib ülesande või seansi kestel, kuid ei salvestata püsivalt. Näiteks agent võib mitme sammulise planeerimisvestluse jooksul koguda fakte ja kasutada neid lõpliku marsruudi koostamiseks.

### Pikaajaline mälu
Eelistused ja faktid, mis püsivad **seansside vahel**. Tagasi tulev kasutaja ei peaks korduvalt kordama oma dieedi piiranguid või reisimise stiili. Pikaajaline mälu toetub tavaliselt välisele andmehoidjale — andmebaasile, failile või vektorindeksile — ja tuuakse agendile tööriistade kaudu.


## Töömäluga seansside abil

Lihtsaim mäluvorm on vestluse sessioon. Kui edastate sama seanssi (loodud `agent.create_session()` abil) järgmistele `agent.run()` kutsumistele, näeb agent kogu selle vestluse ajalugu ja suudab meelde tuletada varasemaid üksikasju.

Loo reisibürooagent ja demonstreerime töömälgu funktsiooni.


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

Agent mäletas eelarvet õigesti, sest mõlemad sõnumid kuuluvad samasse sessiooni. See on **töömälu** — see eksisteerib ainult sessiooni eluea jooksul.

### Mis juhtub uue teema puhul?

Kui me loome **uue** sessiooni, siis agendil puudub mälu eelmise vestluse kohta:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## Pikaajalise Mälu Muster

Kasutaja eelistuste meeldejätmiseks **istungite vahel** vajame püsivat andmehoidlat, mis asub vestluse keerme väliselt. Agent pääseb sellele andmehoidlale ligi läbi **tööriistade** — funktsioonide, mida ta saab kasutada informatsiooni salvestamiseks ja taastamiseks.

Allpool rakendame lihtsa mälus oleva eelistuste salvestamise lahenduse (tootmises kasutaksite selle taga andmebaasi või vektorindeksit) ja teeme selle kättesaadavaks tööriistadena, mida agent saab kasutada.

### Arhitektuur
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Stsenaarium 1 — Esmakordne kasutaja broneerib aastapäeva reisi

Sarah külastab esimest korda. Agendil tuleks tööriistade kaudu salvestada tema eelistused ja kasutada neid hotellide soovitamiseks.


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### Stsenaarium 2 — Sarah naaseb nädalate pärast

Sarah alustab **täiesti uut vestlust** (simuleerides uut sessiooni). Töömällu on tühi, kuid pikaajalises eelistuste andmebaasis on tema info endiselt olemas. Agendil tuleb see üles leida ja kasutada soovituste isikupärastamiseks.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Kokkuvõte

Selles õppetükis õppisite tundma kolme tüüpi agendi mälu ja kuidas neid rakendada Microsoft Agent Frameworki abil:

| Mälu tüüp | MAF mehhanism | Eluaeg |
|---|---|---|
| **Töömäl** | `agent.create_session()` | Üks vestlus |
| **Lühiajaline** | Kogunenud kontekst ühe lõimena | Üks ülesanne / sessioon |
| **Pikaajaline** | Väline andmehoidla, kuhu ligi pääseb `@tool` funktsioonide kaudu | Üle sessioonide |

### Olulisemad punktid
1. **`agent.create_session()`** pakub töömälusisu — agent näeb kogu vestluse ajalugu sessiooni jooksul.
2. **Uued sessioonid kaotavad konteksti** — ilma pikaajalise mäluta agent ei mäleta varasemaid vestlusi.
3. **`@tool` funktsioonid** täidavad lünga — need võimaldavad agendil salvestada ja pärida teavet püsivast andmehoidlast.
4. **Isikupärastamine paraneb aja jooksul** — mida rohkem eelistusi on salvestatud, seda paremad on agendi soovitused.

### Reaalmaailma rakendused
- **Klienditeenindus**: Kliendi ajaloo ja eelistuste meelespidamine
- **Isiklikud assistendid**: Konteksti säilitamine päevade või nädalate ulatuses
- **Tervishoid**: Patsientide teabe ja eelistuste jälgimine
- **E-kaubandus**: Isikupärastatud ostlemine ajaloo põhjal

### Järgmised sammud
- Asenda mäluandmete sõnastik andmebaasi või vektoripoega (nt Azure AI Search)
- Lisa mälu aegumisaeg ajas tundliku info jaoks
- Loo mitme agendiga süsteeme jagatud mäluga
- Uuri Cognee märkmikku teadmiste graafikust toetatud mäluks


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Teatiseave**:
See dokument on tõlgitud kasutades tehisintellekti tõlketeenust [Co-op Translator](https://github.com/Azure/co-op-translator). Kuigi püüame tagada täpsust, palun arvestage, et automaatsed tõlked võivad sisaldada vigu või ebatäpsusi. Originaaldokument selle emakeeles tuleks pidada autoriteetseks allikaks. Kriitilise teabe puhul soovitatakse kasutada professionaalset inimtõlget. Me ei vastuta selle tõlke kasutamisest tulenevate arusaamatuste ega valesti tõlgendamise eest.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
